# KnowGen Multi-Agent System Test (v3)
## Testing Supervisor + Retriever + Generator Agents

This notebook runs the full online pipeline on three source documents:
- `NNHT.pdf`
- `Chương 10 - Giao thoa ánh sáng.pdf`
- `BT CNXHKH.docx`

Pipeline under test: **Supervisor → Retriever → Generator**.

For each query we inspect:
- **Supervisor**: `task_type`, `language`, `intent_summary`, `plan`
- **Retriever**: `rewritten_query`, `retrieved_docs`, `evidence_summary`, `retrieval_strategy`
- **Generator**: `draft`, `generation_metadata` (cited chunks, difficulty distribution for quizzes)

## Section 1: Setup Environment & Paths

In [2]:
import sys
import os
import json
import logging
from pathlib import Path
from dotenv import load_dotenv

# Resolve backend dir regardless of where the notebook is launched from
_here = Path.cwd()
backend_dir = _here
for candidate in [_here, *_here.parents]:
    if candidate.name == 'backend' or (candidate / 'backend').exists():
        backend_dir = candidate if candidate.name == 'backend' else candidate / 'backend'
        break
sys.path.insert(0, str(backend_dir))

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('knowgen.test3')

SOURCE_FILES = [
    r'D:\KnowGen-Project\NNHT.pdf',
    r'D:\KnowGen-Project\Chương 10 - Giao thoa ánh sáng.pdf',
    r'D:\KnowGen-Project\BT CNXHKH.docx',
]
VECTOR_STORE_DIR = backend_dir / 'vector_store' / 'faiss_index'

print(f'Backend dir : {backend_dir}')
print(f'Vector store: {VECTOR_STORE_DIR}')
print('Source files:')
for p in SOURCE_FILES:
    tag = 'OK  ' if os.path.exists(p) else 'MISS'
    print(f'  [{tag}] {p}')

Backend dir : d:\KnowGen-Project\backend
Vector store: d:\KnowGen-Project\backend\vector_store\faiss_index
Source files:
  [OK  ] D:\KnowGen-Project\NNHT.pdf
  [OK  ] D:\KnowGen-Project\Chương 10 - Giao thoa ánh sáng.pdf
  [OK  ] D:\KnowGen-Project\BT CNXHKH.docx


## Section 2: Load Agent Modules (fresh)

In [3]:
# Clear any previously cached versions so we always pick up the latest code
for mod in list(sys.modules):
    if mod.startswith('app.agents') or mod.startswith('app.llm') or mod.startswith('app.ingestion'):
        del sys.modules[mod]

from app.agents.agent_state import AgentState
from app.agents.supervisor_agent import SupervisorAgent, supervisor_node
from app.agents.retriever_agent import RetrieverAgent
from app.agents.generator_agent import GeneratorAgent, generator_node
from app.llm.llm_client import LLMClient

print('Modules loaded:')
print(f'  SupervisorAgent -> {SupervisorAgent}')
print(f'  RetrieverAgent  -> {RetrieverAgent}')
print(f'  GeneratorAgent  -> {GeneratorAgent}')

2026-04-21 11:46:46,793 - knowgen.retriever - INFO - Loading embedding model: intfloat/multilingual-e5-base
2026-04-21 11:46:48,064 - numexpr.utils - INFO - NumExpr defaulting to 16 threads.
2026-04-21 11:46:49,835 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cpu
2026-04-21 11:46:49,835 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: intfloat/multilingual-e5-base
2026-04-21 11:46:50,401 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-21 11:46:50,436 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-base/d128750597153bb5987e10b1c3493a34e5a4502a/modules.json "HTTP/1.1 200 OK"
2026-04-21 11:46:50,701 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 No

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-04-21 11:46:53,438 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-21 11:46:53,473 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-base/d128750597153bb5987e10b1c3493a34e5a4502a/config.json "HTTP/1.1 200 OK"
2026-04-21 11:46:53,752 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-21 11:46:53,775 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multil

Modules loaded:
  SupervisorAgent -> <class 'app.agents.supervisor_agent.SupervisorAgent'>
  RetrieverAgent  -> <class 'app.agents.retriever_agent.RetrieverAgent'>
  GeneratorAgent  -> <class 'app.agents.generator_agent.GeneratorAgent'>


## Section 3: Build FAISS Index for the 3 Source Files

The Retriever needs a populated FAISS index. We rebuild it here from the three
source files so the notebook is self-contained. Set `REBUILD_INDEX = False` to reuse
an existing index.

In [4]:
import time
from app.ingestion.extract import extract_pdf_to_md, extract_docx_to_md
from app.ingestion.transform import transform_documents
from app.ingestion.load import load_documents_to_faiss

REBUILD_INDEX = True  # set False to reuse an existing index

if REBUILD_INDEX:
    print('=' * 80)
    print('RUNNING ETL')
    print('=' * 80)
    extracted = []
    t0 = time.perf_counter()
    for i, path in enumerate(SOURCE_FILES, 1):
        print(f'[{i}/{len(SOURCE_FILES)}] {path}')
        if not os.path.exists(path):
            print('  -> missing, skipping')
            continue
        try:
            if path.lower().endswith('.pdf'):
                doc = extract_pdf_to_md(path)
            elif path.lower().endswith('.docx'):
                doc = extract_docx_to_md(path)
            else:
                print('  -> unsupported, skipping')
                continue
            extracted.append(doc)
            print(f'  extracted {doc.id} ({doc.source_type}, {len(doc.content):,} chars)')
        except Exception as e:
            print(f'  ERROR: {e}')

    if not extracted:
        raise RuntimeError('No documents extracted — cannot continue.')

    print('\nTransforming...')
    transformed = transform_documents(extracted)
    total_chunks = sum(len(d.chunks or []) for d in transformed)
    print(f'  {len(transformed)} docs, {total_chunks} chunks total')

    print('\nLoading into FAISS...')
    load_documents_to_faiss(transformed, str(VECTOR_STORE_DIR))
    print(f'  Index written to {VECTOR_STORE_DIR}')
    print(f'\nETL completed in {time.perf_counter() - t0:.1f}s')
else:
    print(f'Reusing existing index at {VECTOR_STORE_DIR}')

assert os.path.exists(VECTOR_STORE_DIR), 'FAISS index was not produced'

RUNNING ETL
[1/3] D:\KnowGen-Project\NNHT.pdf
Extracting PDF (Markdown): D:\KnowGen-Project\NNHT.pdf
  extracted NNHT.pdf (pdf, 92,078 chars)
[2/3] D:\KnowGen-Project\Chương 10 - Giao thoa ánh sáng.pdf
Extracting PDF (Markdown): D:\KnowGen-Project\Chương 10 - Giao thoa ánh sáng.pdf
  extracted Chương 10 - Giao thoa ánh sáng.pdf (pdf, 15,810 chars)
[3/3] D:\KnowGen-Project\BT CNXHKH.docx
Extracting DOCX (Markdown): D:\KnowGen-Project\BT CNXHKH.docx
  extracted BT CNXHKH.docx (docx, 8,756 chars)

Transforming...


2026-04-21 11:47:46,458 - app.ingestion.transform - INFO - Transforming document: NNHT.pdf
2026-04-21 11:47:46,571 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.
2026-04-21 11:47:57,699 - httpx - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-21 11:47:57,719 - app.ingestion.transform - INFO - Done 'NNHT.pdf': summary=ok, chunks=356
2026-04-21 11:47:57,720 - app.ingestion.transform - INFO - Transforming document: Chương 10 - Giao thoa ánh sáng.pdf
2026-04-21 11:47:57,723 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.
2026-04-21 11:48:05,791 - httpx - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-21 11:48:05,800 - app.ingestion.transform - INFO - Done 'Chương 10 - Giao thoa ánh sáng.pdf': summary=ok, chunks=18
2026-04-21 11:48:05,800 - app.ingestion.t

  3 docs, 399 chunks total

Loading into FAISS...


2026-04-21 11:48:14,511 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-21 11:48:14,548 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-base/d128750597153bb5987e10b1c3493a34e5a4502a/modules.json "HTTP/1.1 200 OK"
2026-04-21 11:48:14,815 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Found"
2026-04-21 11:48:15,074 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Found"
2026-04-21 11:48:15,348 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-04-21 11:48:15,369 - httpx - INFO - HTTP Request: HEAD https://huggingfa

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-04-21 11:48:17,163 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-21 11:48:17,195 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-base/d128750597153bb5987e10b1c3493a34e5a4502a/config.json "HTTP/1.1 200 OK"
2026-04-21 11:48:17,473 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-21 11:48:17,508 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multil

  Index written to d:\KnowGen-Project\backend\vector_store\faiss_index

ETL completed in 110.4s


## Section 4: Define Test Queries

Mix of QA (factual) and quiz prompts across the three documents, in both Vietnamese
and English. Also includes one off-topic query to exercise the insufficient-context
fallback path in the Generator.

In [5]:
TEST_QUERIES = [
    # QA / Vietnamese — CNXHKH
    'Cho biết các đặc điểm chính của chủ nghĩa xã hội khoa học.',
    # QA / Vietnamese — Giao thoa ánh sáng
    'Điều kiện để có hiện tượng giao thoa ánh sáng là gì?',
    # QA / English — NNHT
    'What are the core principles discussed in the NNHT document?',
    # Quiz / Vietnamese — Giao thoa ánh sáng
    'Hãy tạo 3 câu hỏi trắc nghiệm về giao thoa ánh sáng, có mức độ khó khác nhau.',
    # Quiz / Vietnamese — CNXHKH
    'Tạo 5 câu hỏi trắc nghiệm về chủ nghĩa xã hội khoa học dành cho sinh viên.',
    # Off-topic — should hit insufficient-context fallback
    'Explain quantum entanglement in 2 sentences.',
]

for i, q in enumerate(TEST_QUERIES, 1):
    print(f'[{i}] {q}')

[1] Cho biết các đặc điểm chính của chủ nghĩa xã hội khoa học.
[2] Điều kiện để có hiện tượng giao thoa ánh sáng là gì?
[3] What are the core principles discussed in the NNHT document?
[4] Hãy tạo 3 câu hỏi trắc nghiệm về giao thoa ánh sáng, có mức độ khó khác nhau.
[5] Tạo 5 câu hỏi trắc nghiệm về chủ nghĩa xã hội khoa học dành cho sinh viên.
[6] Explain quantum entanglement in 2 sentences.


## Section 5: Instantiate Agents

In [6]:
supervisor = SupervisorAgent()
retriever  = RetrieverAgent(vector_store_dir=str(VECTOR_STORE_DIR))
generator  = GeneratorAgent()

print('Agents ready.')

2026-04-21 11:48:51,613 - knowgen.retriever - INFO - Loading embedding model: intfloat/multilingual-e5-base
2026-04-21 11:48:51,615 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cpu
2026-04-21 11:48:51,616 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: intfloat/multilingual-e5-base
2026-04-21 11:48:51,996 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-21 11:48:52,027 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-base/d128750597153bb5987e10b1c3493a34e5a4502a/modules.json "HTTP/1.1 200 OK"
2026-04-21 11:48:52,308 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Found"
2026-04-21 11:48:52,570 - httpx - INFO - HTTP Request: HEAD https://huggin

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-04-21 11:48:54,750 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-21 11:48:54,781 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-base/d128750597153bb5987e10b1c3493a34e5a4502a/config.json "HTTP/1.1 200 OK"
2026-04-21 11:48:55,055 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-21 11:48:55,086 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multil

Agents ready.


## Section 6: Run the Full Pipeline

For each query we run Supervisor → Retriever → Generator and collect every
intermediate output in `pipeline_results` for later inspection.

In [7]:
pipeline_results = []

for i, query in enumerate(TEST_QUERIES, 1):
    print('=' * 80)
    print(f'QUERY {i}/{len(TEST_QUERIES)}: {query}')
    print('=' * 80)

    state = {'user_query': query}

    # 1) Supervisor
    try:
        sup_out = supervisor.run(state)
        state.update(sup_out)
        print(f"  [Supervisor] task_type={sup_out.get('task_type')} language={sup_out.get('language')}")
        print(f"               intent: {sup_out.get('intent_summary')}")
    except Exception as e:
        print(f'  [Supervisor] ERROR: {e}')
        pipeline_results.append({'query': query, 'error': f'supervisor: {e}'})
        continue

    # 2) Retriever
    try:
        ret_out = retriever.run(state)
        state.update(ret_out)
        n_docs = len(ret_out.get('retrieved_docs') or [])
        print(f'  [Retriever]  retrieved={n_docs} docs')
        print(f"               rewritten: {(ret_out.get('rewritten_query') or '')[:100]}")
    except Exception as e:
        print(f'  [Retriever] ERROR: {e}')
        pipeline_results.append({'query': query, 'error': f'retriever: {e}', 'supervisor': sup_out})
        continue

    # 3) Generator
    try:
        gen_out = generator.run(state)
        state.update(gen_out)
        meta = gen_out.get('generation_metadata', {})
        draft = gen_out.get('draft', '')
        print(f"  [Generator]  task={meta.get('task_type')} used_chunks={meta.get('used_chunks')}")
        if meta.get('task_type') == 'quiz':
            print(f"               num_questions={meta.get('num_questions')} "
                  f"difficulty={meta.get('difficulty_distribution')}")
        print(f'               draft preview: {draft[:200]}...')
    except Exception as e:
        print(f'  [Generator] ERROR: {e}')
        pipeline_results.append({'query': query, 'error': f'generator: {e}',
                                 'supervisor': sup_out, 'retriever': ret_out})
        continue

    pipeline_results.append({
        'query': query,
        'supervisor': sup_out,
        'retriever': ret_out,
        'generator': gen_out,
    })

print(f'\nPipeline runs completed: {len(pipeline_results)}')

2026-04-21 11:48:59,261 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.


QUERY 1/6: Cho biết các đặc điểm chính của chủ nghĩa xã hội khoa học.


2026-04-21 11:49:01,684 - httpx - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-21 11:49:01,690 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.


  [Supervisor] task_type=qa language=vi
               intent: Người dùng muốn biết các đặc điểm chính của chủ nghĩa xã hội khoa học.


2026-04-21 11:49:06,712 - httpx - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-21 11:49:06,720 - knowgen.retriever - INFO - Query rewritten: 'Cho biết các đặc điểm chính của chủ nghĩ…' (58 chars) → 'Chủ nghĩa xã hội khoa học đặc điểm chính…' (181 chars)
2026-04-21 11:49:06,720 - knowgen.retriever - INFO - Searching 20 candidates…
2026-04-21 11:49:06,823 - knowgen.retriever - INFO - Found 20 candidates
2026-04-21 11:49:06,824 - knowgen.retriever - INFO - Doc-relevance gate: 3/3 doc(s) passed, 20/20 chunks retained.
2026-04-21 11:49:07,064 - knowgen.retriever - INFO - After ranking & filtering: 2 documents
2026-04-21 11:49:07,065 - knowgen.retriever - INFO - Retrieval complete: 2 docs, 2 evidence items
2026-04-21 11:49:07,066 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.


...

  [Supervisor] ERROR: Error calling model 'models/gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 48.118396018s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

2026-04-21 11:51:13,583 - httpx - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 429 Too Many Requests"
2026-04-21 11:51:13,585 - google_genai._api_client - INFO - Retrying google.genai._api_client.BaseApiClient._request_once in 1.2525734549477368 seconds as it raised ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 47.796413367s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'http

  [Supervisor] ERROR: Error calling model 'models/gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 11.698402806s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

2026-04-21 11:51:49,996 - httpx - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 429 Too Many Requests"
2026-04-21 11:51:49,998 - google_genai._api_client - INFO - Retrying google.genai._api_client.BaseApiClient._request_once in 1.4607278430087598 seconds as it raised ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 11.382632828s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'http

  [Supervisor] ERROR: Error calling model 'models/gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 36.421596639s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

## Section 7: Inspect Supervisor Outputs

In [8]:
for i, r in enumerate(pipeline_results, 1):
    print(f"--- Query {i}: {r['query']}")
    sup = r.get('supervisor') or {}
    if not sup:
        print('  (no supervisor output)')
        continue
    print(f"  task_type      : {sup.get('task_type')}")
    print(f"  language       : {sup.get('language')}")
    print(f"  intent_summary : {sup.get('intent_summary')}")
    plan = sup.get('plan') or {}
    print(f"  plan.steps     : {plan.get('steps')}")
    print(f"  context_needed : {plan.get('context_needed')}")
    print()

--- Query 1: Cho biết các đặc điểm chính của chủ nghĩa xã hội khoa học.
  task_type      : qa
  language       : vi
  intent_summary : Người dùng muốn biết các đặc điểm chính của chủ nghĩa xã hội khoa học.
  plan.steps     : ["Bước 1: Tìm kiếm và truy xuất các tài liệu liên quan đến 'chủ nghĩa xã hội khoa học' và 'đặc điểm' của nó từ kho tri thức.", 'Bước 2: Phân tích và tổng hợp các đặc điểm chính của chủ nghĩa xã hội khoa học từ các thông tin đã truy xuất.', 'Bước 3: Trả lời câu hỏi của người dùng bằng tiếng Việt, liệt kê các đặc điểm chính đã tổng hợp.']
  context_needed : chủ nghĩa xã hội khoa học, đặc điểm, tính chất

--- Query 2: Điều kiện để có hiện tượng giao thoa ánh sáng là gì?
  task_type      : qa
  language       : vi
  intent_summary : The user wants to know the conditions required for light interference.
  plan.steps     : ["Step 1: Use the retriever agent to search for documents containing information about 'giao thoa ánh sáng' and 'điều kiện giao thoa'.", 'Step 2: Use 

## Section 8: Inspect Retriever Outputs

In [9]:
for i, r in enumerate(pipeline_results, 1):
    print(f"--- Query {i}: {r['query']}")
    ret = r.get('retriever') or {}
    if not ret:
        print('  (no retriever output)')
        continue
    docs = ret.get('retrieved_docs') or []
    print(f"  rewritten_query : {ret.get('rewritten_query', '')}")
    print(f"  strategy        : {ret.get('retrieval_strategy')}")
    print(f'  #docs           : {len(docs)}')
    for j, doc in enumerate(docs, 1):
        meta = doc.metadata or {}
        sim  = meta.get('similarity_score', 0.0)
        src  = meta.get('source_type', '?')
        title = meta.get('title', '')
        print(f'    [Doc {j}] sim={sim:.3f} src={src} title={title[:60]}')
        preview = doc.page_content[:160].replace('\n', ' ')
        print(f'             {preview}...')
    ev = ret.get('evidence_summary') or []
    print(f'  evidence ({len(ev)}):')
    for e in ev:
        print(f'    - {e[:160]}')
    print()

--- Query 1: Cho biết các đặc điểm chính của chủ nghĩa xã hội khoa học.
  rewritten_query : Chủ nghĩa xã hội khoa học đặc điểm chính tính chất đặc trưng nét chính khía cạnh thuộc tính lý luận Marx Engels Mác-Lênin học thuyết Mác nguyên lý cơ bản bản chất nội dung quan điểm
  strategy        : {'total_retrieved': 2, 'top_k': 5, 'confidence_threshold': 0.7, 'ranking_signals': ['similarity', 'source_type', 'chunk_position'], 'sources': ['unknown', 'docx']}
  #docs           : 2
    [Doc 1] sim=0.777 src=docx title=
             passage: đúng. chủ nghĩa xã hội khoa học có chức năng hướng dẫn giai cấp công nhân, nông dân và trí thức trong quá trình cách mạng nhằm xây dựng chủ nghĩa xã hộ...
    [Doc 2] sim=0.774 src=? title=
             passage: Summary of NNHT.pdf: Tài liệu là giáo trình "Ngôn ngữ hình thức & ôtômát" của Khoa Công nghệ thông tin, Trường Đại học Bách khoa, Đại học Đà Nẵng, nhằm...
  evidence (2):
    - passage: đúng
    - passage: Summary of NNHT

--- Query 2: Điều kiện để

## Section 9: Inspect Generator Outputs

For QA tasks the draft is plain text; for quiz tasks the draft is a JSON string we
parse here to show each question cleanly.

In [10]:
for i, r in enumerate(pipeline_results, 1):
    print('=' * 80)
    print(f"Query {i}: {r['query']}")
    print('=' * 80)

    gen = r.get('generator') or {}
    if not gen:
        print('  (no generator output)')
        continue

    meta = gen.get('generation_metadata', {})
    draft = gen.get('draft', '')

    print(f"task_type      : {meta.get('task_type')}")
    print(f"language       : {meta.get('language')}")
    print(f"used_chunks    : {meta.get('used_chunks')}")
    print(f"revision_count : {meta.get('revision_count')}")
    if meta.get('reason'):
        print(f"reason         : {meta.get('reason')}")

    if meta.get('task_type') == 'quiz':
        print(f"num_questions  : {meta.get('num_questions')}")
        print(f"difficulty_dist: {meta.get('difficulty_distribution')}")
        print('-' * 80)
        try:
            payload = json.loads(draft)
        except json.JSONDecodeError:
            print('  (draft is not valid JSON)')
            print(draft[:800])
            continue
        for q in payload.get('questions', []):
            print(f"  {q.get('id')} [{q.get('difficulty')}/{q.get('bloom_level')}] "
                  f"(chunk={q.get('source_chunk_index')})")
            print(f"    Q: {q.get('question')}")
            for letter, text in (q.get('choices') or {}).items():
                marker = '*' if letter == q.get('correct_answer') else ' '
                print(f'    {marker}{letter}. {text}')
            if q.get('explanation'):
                print(f"    why: {q.get('explanation')}")
            print()
    else:
        print('-' * 80)
        print('DRAFT:')
        print(draft)
    print()

Query 1: Cho biết các đặc điểm chính của chủ nghĩa xã hội khoa học.
task_type      : qa
language       : vi
used_chunks    : [0]
revision_count : 0
--------------------------------------------------------------------------------
DRAFT:
Chủ nghĩa xã hội khoa học có chức năng hướng dẫn giai cấp công nhân, nông dân và trí thức trong quá trình cách mạng nhằm xây dựng chủ nghĩa xã hội và tiến tới chủ nghĩa cộng sản [Doc 1]. Nó cung cấp lý luận, phương pháp và định hướng cho các giai cấp này trong quá trình đấu tranh cách mạng và phát triển xã hội [Doc 1]. Tuy nhiên, chủ nghĩa xã hội khoa học cũng có hạn chế là không chỉ ra được sứ mệnh lịch sử của giai cấp công nhân [Doc 1].

Query 2: Điều kiện để có hiện tượng giao thoa ánh sáng là gì?
task_type      : qa
language       : vi
used_chunks    : [0]
revision_count : 0
--------------------------------------------------------------------------------
DRAFT:
Để có hiện tượng giao thoa ánh sáng, các sóng ánh sáng phải thỏa mãn hai điều kiện chính. 

## Section 10: Summary Statistics

In [11]:
from collections import Counter

total  = len(pipeline_results)
sup_ok = sum(1 for r in pipeline_results if r.get('supervisor'))
ret_ok = sum(1 for r in pipeline_results if (r.get('retriever') or {}).get('retrieved_docs'))
gen_ok = sum(1 for r in pipeline_results if r.get('generator'))

task_dist = Counter((r.get('supervisor') or {}).get('task_type') for r in pipeline_results)
lang_dist = Counter((r.get('supervisor') or {}).get('language')  for r in pipeline_results)

empty_ctx = sum(
    1 for r in pipeline_results
    if ((r.get('generator') or {}).get('generation_metadata') or {}).get('reason') == 'insufficient_context'
)
quiz_runs = [
    r for r in pipeline_results
    if ((r.get('generator') or {}).get('generation_metadata') or {}).get('task_type') == 'quiz'
]

print(f'Total queries       : {total}')
print(f'Supervisor OK       : {sup_ok}/{total}')
print(f'Retriever returned  : {ret_ok}/{total} (non-empty)')
print(f'Generator produced  : {gen_ok}/{total}')
print(f'Empty-context drafts: {empty_ctx}')
print(f'Quiz runs           : {len(quiz_runs)}')
print(f'task_type dist      : {dict(task_dist)}')
print(f'language dist       : {dict(lang_dist)}')

if quiz_runs:
    print('\nQuiz detail:')
    for r in quiz_runs:
        meta = r['generator']['generation_metadata']
        print(f"  - {r['query'][:60]}...")
        print(f"      n={meta.get('num_questions')} "
              f"diff={meta.get('difficulty_distribution')} "
              f"chunks={meta.get('used_chunks')}")

Total queries       : 6
Supervisor OK       : 3/6
Retriever returned  : 2/6 (non-empty)
Generator produced  : 3/6
Empty-context drafts: 1
Quiz runs           : 0
task_type dist      : {'qa': 3, None: 3}
language dist       : {'vi': 2, 'en': 1, None: 3}
